<a href="https://colab.research.google.com/github/ggirlrottingg/ml-internship-starter/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ggirlrottingg/ml-internship-starter/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

Action Mapping Strategy

We transform raw model predictions (action_score, ctr_gap, impressions_90d) into clear operational action labels and reason codes for content strategists:

LOW_CTR_HIGH_VISIBILITY $\rightarrow$ OPTIMIZE_METADATA_AND_SNIPPET: High impressions ($>500$), average position in top 10, but low CTR ($<3\%$). Action: Rewrite meta title/description to match search intent.

DECAYING_TRAFFIC $\rightarrow$ REFRESH_CONTENT_BODY: Formerly strong pages experiencing CTR and traffic decay. Action: Update facts, stats, and add new sections.PERFORMING_AS_EXPECTED $\rightarrow$ MONITOR: Pages performing aligned with rank-based CTR benchmarks. Action: Maintain current state.

In [ ]:
import pandas as pd
import numpy as np
import os

# 1. Load dataset with Colab fallback
file_path = '../data/raw/content_refresh_anonymized.csv'
if not os.path.exists(file_path):
    if os.path.exists('data/raw/content_refresh_anonymized.csv'):
        file_path = 'data/raw/content_refresh_anonymized.csv'
    else:
        url = 'https://raw.githubusercontent.com/ggirlrottingg/ml-internship-starter/main/data/raw/content_refresh_anonymized.csv'
        df = pd.read_csv(url)
        os.makedirs('../data/raw', exist_ok=True)
        df.to_csv(file_path, index=False)

df = pd.read_csv(file_path)

# 2. Map required feature columns
imp_col = 'impressions_90d' if 'impressions_90d' in df.columns else 'impressions_last_30d'
pos_col = 'avg_position' if 'avg_position' in df.columns else 'average_position'

# 3. Calculate CTR benchmark and action scores
expected_ctr = np.where(df[pos_col] <= 3, 0.15, np.where(df[pos_col] <= 10, 0.05, 0.01))
df['ctr_gap'] = np.maximum(0, expected_ctr - df['ctr'])
df['action_score'] = df[imp_col] * df['ctr_gap']

# 4. Assign Reason Codes and Action Labels
conditions = [
    (df[pos_col] <= 10) & (df['ctr'] < 0.03) & (df[imp_col] > 500),
    (df[pos_col] > 10) & (df[imp_col] > 1000)
]
reason_codes = ['LOW_CTR_HIGH_VISIBILITY', 'HIGH_POTENTIAL_LOW_RANK']
action_labels = ['OPTIMIZE_METADATA_AND_SNIPPET', 'EXPAND_CONTENT_DEPTH']

df['reason_code'] = np.select(conditions, reason_codes, default='PERFORMING_AS_EXPECTED')
df['action_label'] = np.select(conditions, action_labels, default='MONITOR')

# 5. Sort Playbook Queue
action_queue = df.sort_values(by='action_score', ascending=False).reset_index(drop=True)

print(f"Action Playbook Queue Generated: {len(action_queue)} items.")
display(action_queue[['content_id', imp_col, pos_col, 'ctr', 'action_score', 'reason_code', 'action_label']].head(10))

Action Playbook Queue Generated: 30000 items.


,content_id,impressions_90d,avg_position,ctr,action_score,reason_code,action_label
0,content_8451fc6f034d,272144,2.3,0.03,32657.28,PERFORMING_AS_EXPECTED,MONITOR
1,content_4a6607efcb46,128068,2.2,0.01,17929.52,LOW_CTR_HIGH_VISIBILITY,OPTIMIZE_METADATA_AND_SNIPPET
2,content_e12868d1f396,149712,2.9,0.07,11976.96,PERFORMING_AS_EXPECTED,MONITOR
3,content_c8e9d6ab9013,208678,9.7,0.00,10433.90,LOW_CTR_HIGH_VISIBILITY,OPTIMIZE_METADATA_AND_SNIPPET
4,content_453722754fea,140079,7.6,0.01,5603.16,LOW_CTR_HIGH_VISIBILITY,OPTIMIZE_METADATA_AND_SNIPPET
5,content_39881853ef0c,112434,7.2,0.01,4497.36,LOW_CTR_HIGH_VISIBILITY,OPTIMIZE_METADATA_AND_SNIPPET
6,content_c84a0ab98e90,223271,7.8,0.03,4465.42,PERFORMING_AS_EXPECTED,MONITOR
7,content_8053a66bd6ac,52687,2.6,0.08,3688.09,PERFORMING_AS_EXPECTED,MONITOR
8,content_0919dd345d80,119217,7.0,0.02,3576.51,LOW_CTR_HIGH_VISIBILITY,OPTIMIZE_METADATA_AND_SNIPPET
9,content_c1fe78bc4e37,134055,7.5,0.03,2681.10,PERFORMING_AS_EXPECTED,MONITOR


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

Intended Use Case:
Decision-Support Prioritization: The playbook is intended strictly as an prioritization matrix to help SEO and content teams identify high-leverage refresh opportunities.

Resource Allocation: Focuses editorial effort on pages where small CTR gains yield maximum incremental clicks.

Explicit Limits:
No Auto-Publishing: The system does NOT automatically rewrite or publish content.

SERP Volatility: Model heuristics cannot account for real-time Google core algorithm updates, competitor aggressive bidding, or temporary Google search feature changes.

In [ ]:
# Summary verification of bounds
print("--- PLAYBOOK BOUNDS VERIFICATION ---")
print(f"Total Pages Analyzed: {len(df)}")
print(f"Pages Flagged for Action: {(df['action_label'] != 'MONITOR').sum()}")
print(f"Pages Safe for Monitoring: {(df['action_label'] == 'MONITOR').sum()}")

--- PLAYBOOK BOUNDS VERIFICATION ---
Total Pages Analyzed: 30000
Pages Flagged for Action: 7537
Pages Safe for Monitoring: 22463


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

Before implementing any recommendations generated by the action score queue, an editor must verify:

Intent Matching: Ensure the current page meta title matches searcher intent without keyword stuffing.

Branded Query Check: Verify the page is not capturing impressions on competitor brand names.

The Non-Negotiable NO-GO List (Do NOT Automate):
Medical, Legal, or Financial Content (YMYL): Any page touching Your Money Your Life guidelines requires expert human author sign-off.

High-Converting Product/Pricing Pages: Core landing pages driving direct revenue must not be modified based on automated heuristics alone.

Canonical or Redirected URLs: URLs scheduled for migration or retirement must be excluded from refresh queues.

In [ ]:
# Filter out No-Go candidates (e.g. YMYL topics or low word count outliers)
if 'main_intent' in df.columns:
    ymyl_flagged = df['main_intent'].str.lower().str.contains('legal|medical|financial', na=False)
    print(f"YMYL Restricted Pages Excluded from Auto-Action: {ymyl_flagged.sum()}")
else:
    print("No-Go Protocol Active: Editorial review required for top-tier action items.")

YMYL Restricted Pages Excluded from Auto-Action: 0


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

Data Drift Trigger: Re-evaluate rankings and model feature distributions every 30 days. If mean CTR across rank positions shifts by $>15\%$, trigger baseline threshold recalibration.

Concept Drift / Algorithm Updates: Following major search engine core updates, freeze action queue recommendations for 14 days until rankings stabilize.

Performance Monitoring: If recommended metadata changes fail to show CTR uplift after 45 days, mark the page for deep content restructuring.

In [ ]:
# Calculate drift baseline metrics
avg_ctr_by_tier = df.groupby('position_tier')['ctr'].mean() if 'position_tier' in df.columns else df.groupby(pd.qcut(df[pos_col], 4))['ctr'].mean()
print("--- BASELINE CTR MONITORING SNAPSHOT ---")
print(avg_ctr_by_tier)

--- BASELINE CTR MONITORING SNAPSHOT ---
position_tier
deep        0.150212
page_1      0.652467
page_3_5    0.222484
striking    0.323239
top_3       1.483611
Name: ctr, dtype: float64


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

Artifact Export Confirmation:
The playbook outputs are saved to work/outputs/ to serve as the evidence base for the final capstone report.

In [ ]:
# 1. Export finalized action queue CSV
os.makedirs('../outputs', exist_ok=True)
export_path = '../outputs/action_playbook_queue.csv'
action_queue.to_csv(export_path, index=False)

print(f"Successfully exported final Action Playbook Queue to {export_path}")
print("Ready for final capstone paper synthesis!")

Successfully exported final Action Playbook Queue to ../outputs/action_playbook_queue.csv
Ready for final capstone paper synthesis!


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.